# W05 Exercise: Confusion matrices, thresholds, and prevalence

Machine Learning in Engineering Science | NCKU / LAiMM | Fall 2026

**Date:** 2026-10-05. **Exercise time:** 40 minutes within the class/practice allocation, not extra time after class.

**Classroom formative practice. No new weekly graded submission is created.**

## How to work independently
1. In Colab, save a copy to your Drive and use a CPU runtime. In Jupyter, open this notebook and select a Python kernel.
2. Run the complete worked walkthrough below from top to bottom. Read every Markdown checkpoint; the walkthrough is included so no previous notebook state is needed.
3. At the Independent Exercise section, write each prediction in the response cell before running its experiment.
4. Run the starter experiment, make the requested controlled change, and record the actual result. Edit only the declared factor; preserve split rules and preprocessing.
5. Explain one mechanism and one limitation. Export the evidence JSON and discuss it with a partner.

The core exercises need no paid API, private dataset, or GPU. The agent units use clearly labeled offline simulators, not real LLM calls.

**Reproducibility boundary:** the original guided notebooks were previously executed; the new extension cells have not yet been execution-tested in this revision. Report unexpected behavior rather than claiming a check passed.


# Week 05: Validation, thresholds, precision, and recall

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Purpose:** a fully worked, self-paced in-class practice with an optional extension. Weeks 11 and 18 are optional examination-readiness materials. These notebooks do not create weekly graded assignments.

**Before running:** choose a Python 3 kernel in Jupyter or a CPU runtime in Colab. Run every cell from top to bottom with Shift+Enter. Setup installs missing free packages. No API key or external dataset download is required.

**Learning target:** complete the calculation, inspect intermediate values, and explain why the protocol supports a limited claim.

**Time:** about 25-45 minutes including reading, prediction, execution, and explain-back.


## 1. Set up and record the environment

Keep the seed fixed for the first run. Record actual outputs and package versions rather than copying a claimed performance number.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** The environment cell must finish before any analysis cell. Restart and run all if a variable is missing.


## 2. Reserve validation and test separately

Class 1 denotes the less frequent synthetic event. Select a decision threshold using validation only; preserve test for one final comparison.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score,recall_score,f1_score,confusion_matrix
X,y=make_classification(n_samples=800,n_features=8,n_informative=4,weights=[.85,.15],random_state=SEED)
Xtr,Xrest,ytr,yrest=train_test_split(X,y,test_size=.4,stratify=y,random_state=SEED)
Xva,Xte,yva,yte=train_test_split(Xrest,yrest,test_size=.5,stratify=yrest,random_state=SEED)
model=LogisticRegression(max_iter=300).fit(Xtr,ytr)
pv=model.predict_proba(Xva)[:,1]


**Checkpoint.** There are 480 training, 160 validation, and 160 test rows. No threshold has been selected using test outcomes.


## 3. Select a threshold on validation

For illustration choose the validation threshold with highest F1 among a fixed grid. A real safety specification may instead demand a minimum recall and account for uncertainty.


In [ ]:
thresholds=np.linspace(.05,.95,19)
scores=np.array([f1_score(yva,pv>=t,zero_division=0) for t in thresholds])
threshold=float(thresholds[scores.argmax()])
plt.plot(thresholds,scores,"o-"); plt.xlabel("Decision threshold"); plt.ylabel("Validation F1"); plt.show()
print("Selected threshold:",threshold)


**Checkpoint.** The chosen number is specific to this dataset and protocol. It is not a universal optimal threshold.


## 4. Explain every confusion-matrix cell

Recall is TP/(TP+FN), also called sensitivity. Specificity is TN/(TN+FP), and precision is TP/(TP+FP). If the denominator is zero, the metric is undefined; software zero_division=0 is a reporting convention.


In [ ]:
pt=model.predict_proba(Xte)[:,1]; predicted=(pt>=threshold)
tn,fp,fn,tp=confusion_matrix(yte,predicted,labels=[0,1]).ravel()
summary={"threshold":threshold,"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn),
         "precision":float(precision_score(yte,predicted,zero_division=0)),
         "recall_sensitivity":float(recall_score(yte,predicted,zero_division=0)),
         "specificity":float(tn/(tn+fp)),"F1":float(f1_score(yte,predicted,zero_division=0))}
print(summary)


**Checkpoint.** Identify a false negative and a false positive in words. State which cost your application should care about and why.


## 6. Preserve your evidence

Export a JSON record with a seed, package version, and measured result. Download the artifacts directory before leaving Colab; runtime storage is temporary.


In [ ]:
record={"week":5,"seed":SEED,"numpy":np.__version__,"python":platform.python_version(),"result":summary}
(ARTIFACTS/"week-05-evidence.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


**Checkpoint.** The evidence file must match the result you actually obtained. A saved screenshot alone does not permit rerunning the calculation.


## 7. Try one change and explain it
1. Predict the effect of one change before editing code. Change one parameter, input, or boundary condition only.
2. Rerun all cells in a fresh kernel. Keep the original evidence record and compare the observed difference.
3. Explain the result using the mechanism, not just the final number.
4. Name one assumption that limits the claim, and a new test that could challenge it.

**Troubleshooting:** if installation fails, restore network access; if a variable is undefined, run earlier cells; if your output differs, compare package versions, seed, and data construction. Do not repeatedly edit code until a preferred score appears.

**Completion criterion:** one executed notebook, one evidence JSON, and a brief claim/evidence/limitation statement.

### Reference documentation
- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- [Course repository](https://github.com/CHjonnyYu/MLES-Fall-2026)


## Independent Exercise: W05

Complete the three tasks below after the walkthrough. The starter cells produce observations, not your written interpretation. Fill the response fields in the next cell. Save separate records for the original run and any controlled change.

In [ ]:
exercise_id="W05"
observations={}
responses={
    'task1_prediction':'', 'task1_explanation':'',
    'task2_prediction':'', 'task2_explanation':'',
    'task3_prediction':'', 'task3_explanation':'',
    'controlled_change':'', 'limitation':'', 'ai_use_and_checks':''
}
# Fill these strings before and after each experiment. Use 'none' if no AI assistance was used.
print('Exercise record initialized:', exercise_id)

### Task 1: Explain every denominator

Compute precision, recall/sensitivity, specificity, and accuracy from the counts below. Name the population in each denominator before running.

**Steps**
1. Write your prediction in `responses['task1_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task1_explanation']`; do not merely repeat a score.

**Checkpoint:** Recall is 0.9, precision is 18/26, specificity is 0.9, and accuracy is 0.9. Precision and recall answer different questions.

In [ ]:
tp,fn,fp,tn=18,2,8,72
observations['task1']={'precision':tp/(tp+fp),'recall_sensitivity':tp/(tp+fn),'specificity':tn/(tn+fp),'accuracy':(tp+tn)/(tp+tn+fp+fn)}
print(observations['task1'])

### Task 2: Choose using validation, not test

The following labels and probabilities are a validation set. Compare two thresholds, describe the cost of a false negative, and declare a choice before consulting any test set.

**Steps**
1. Write your prediction in `responses['task2_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task2_explanation']`; do not merely repeat a score.

**Checkpoint:** At 0.3 there are no false negatives and two false positives. At 0.7 there is one false negative and no false positives. Explain the tradeoff.

In [ ]:
from sklearn.metrics import confusion_matrix
validation_y=np.array([0,0,0,1,1,1])
validation_probability=np.array([.1,.35,.6,.4,.7,.9])
observations['task2']={}
for threshold_demo in [.3,.7]:
    cm=confusion_matrix(validation_y,validation_probability>=threshold_demo,labels=[0,1])
    observations['task2'][str(threshold_demo)]=cm.tolist()
print(observations['task2'])

### Task 3: Change prevalence without changing rates

Assume sensitivity 0.9 and false-positive rate 0.1. Compare expected counts in 1000 cases at prevalence 0.1 and 0.01. These are mathematical expectations, not measured model results.

**Steps**
1. Write your prediction in `responses['task3_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task3_explanation']`; do not merely repeat a score.

**Checkpoint:** Precision falls from 0.5 to about 0.0833 even though sensitivity and false-positive rate are held fixed.

In [ ]:
observations['task3']=[]
for prevalence in [.1,.01]:
    expected_tp=1000*prevalence*.9
    expected_fp=1000*(1-prevalence)*.1
    observations['task3'].append({'prevalence':prevalence,'expected_TP':expected_tp,'expected_FP':expected_fp,'expected_precision':expected_tp/(expected_tp+expected_fp)})
print(observations['task3'])

### Finish: preserve and explain

Fill all response strings, then run the export cell. A draft file is intentionally labeled incomplete; a complete response record is not automatically a scientifically correct result. Download the JSON before the Colab session ends. Share it with a partner for explain-back, not as an additional weekly submission.

**Exit questions:** What changed? Which observation supports your explanation? What does this experiment not establish?

In [ ]:
missing_responses=[key for key,value in responses.items() if not str(value).strip()]
exercise_record={'exercise':exercise_id,'status':'draft' if missing_responses else 'ready for peer discussion','responses':responses,'observations':observations,'missing_responses':missing_responses,'environment':{'python':platform.python_version(),'numpy':np.__version__},'data_scope':'course-provided or synthetic teaching data'}
exercise_folder=Path('artifacts')/'exercises'; exercise_folder.mkdir(parents=True,exist_ok=True)
exercise_path=exercise_folder/(exercise_id+'-evidence.json')
exercise_path.write_text(json.dumps(exercise_record,indent=2,default=lambda value:value.tolist() if hasattr(value,'tolist') else str(value)))
print('Saved:',exercise_path)
print('Status:',exercise_record['status'])
print('Fields still to complete:',missing_responses)